In [133]:
import random

from music21 import stream, tempo, meter, key, clef, note, articulations, expressions, spanner, instrument
from music21.clef import clefFromString

In [135]:
bpm = 120
num_sharps = 0
time_signature = "4/4"
num_dice_rolls = 100
part_size = 4

PartClefs = {
    0: "Treble",
    1: "Treble",
    2: "Alto",
    3: "Bass",
}
MiddleLineNotes = {
    "Treble": 71,
    "Alto": 60,
    "Bass": 53,
}

PartInstruments = {
    0: instrument.Violin(),
    1: instrument.Violin(),
    2: instrument.Viola(),
    3: instrument.Violoncello(),
}

make_solid_gliss = False
make_wavy_gliss = False

In [136]:
# Create a Score and a Part
score = stream.Score()
score.append(tempo.MetronomeMark(number=bpm))
parts = []
for i in range(part_size):
    parts.append(stream.Part())
    if part_size <= len(PartClefs):
        parts[i].append(clef.clefFromString(PartClefs[i]))
        parts[i].append(PartInstruments[i])
        setattr(parts[i], "middle_line_note", MiddleLineNotes[PartClefs[i]])
    else:
        parts[i].append(clef.TrebleClef())
        parts[i].append(instrument.Piano())

    parts[i].append(meter.TimeSignature(time_signature))
    parts[i].append(key.KeySignature(num_sharps))


In [140]:
def diceroll(sides, exploding=False):
    dice_roll = random.randint(1, sides)
    total = dice_roll

    if exploding:
        while dice_roll == sides:
            dice_roll = random.randint(1, sides)
            total += dice_roll

    return total

def makenote(part):
    # Note Duration
    longer = True if diceroll(12, False) > 5 else False
    note_length = diceroll(4, False)
    if not longer:
        note_length *= 0.25

    # Note Pitch / rest
    if diceroll(6, False) <= 1:
        n = note.Rest(quarterLength=note_length)
        return n
    else:
        note_pitch = part.middle_line_note
        sign = 1 if random.randint(0, 1) == 0 else -1
        note_pitch += sign * diceroll(8, True)
        n = note.Note(note_pitch, quarterLength=note_length)

    # Articulation
    make_solid_gliss = False
    make_wavy_gliss = False

    artic = diceroll(20, False)
    if artic == 1:
        n.articulations.append(articulations.Accent())
    elif artic == 2:
        n.articulations.append(articulations.Staccato())
    elif artic == 3:
        n.articulations.append(articulations.Staccatissimo())
    elif artic == 4:
        n.articulations.append(articulations.Tenuto())
    elif artic == 5:
        n.articulations.append(articulations.Accent())
        n.articulations.append(articulations.Staccato())
    elif artic == 6:
        trem = expressions.Tremolo()
        trem.measured = True
        trem.number = 1
        n.expressions.append(trem)
    elif artic == 7:
        trem = expressions.Tremolo()
        trem.measured = True
        trem.number = 2
        n.expressions.append(trem)
    elif artic == 8:
        trem = expressions.Tremolo()
        trem.measured = False
        trem.number = 3
        n.expressions.append(trem)
    elif artic == 9:
        trem = expressions.Tremolo()
        trem.measured = True
        trem.number = 4
        n.expressions.append(trem)
    elif artic == 10:
        n.expressions.append(expressions.Turn())
        expressions.Turn().realize(n)
    elif artic == 11:
        n.expressions.append(expressions.InvertedTurn())
        expressions.InvertedTurn().realize(n)
    elif artic == 12:
        trill = expressions.Trill()
        trill.direction = "up"
        n.expressions.append(trill)
    elif artic == 13:
        trill = expressions.Trill()
        trill.direction = "down"
        n.expressions.append(trill)
    elif artic == 14:
        n.expressions.append(expressions.Mordent())
        expressions.Mordent().realize(n)
    elif artic == 15:
        n.articulations.append(articulations.StrongAccent())
    elif artic == 16:
        n.articulations.append(articulations.StrongAccent())
        n.articulations.append(articulations.Staccato())
    elif artic == 17:
        n.articulations.append(articulations.StrongAccent())
        n.articulations.append(articulations.Tenuto())
    elif artic == 20:
        n.articulations.append(articulations.Tenuto())
        n.articulations.append(articulations.Staccato())
    elif artic == 18:
        make_solid_gliss = True

    elif artic == 19:
        make_wavy_gliss = True

    return n


In [141]:
def export():
    for i in range(num_dice_rolls):
        for part in parts:
            part.append(makenote(part))
            if make_solid_gliss:
                gliss = spanner.Glissando()
                gliss.lineType = "solid"
                part.append(spanner.Glissando())
            if make_wavy_gliss:
                gliss = spanner.Glissando()
                gliss.linetype = "wavy"
                part.append(spanner.Glissando())

    for part in parts:
        score.insert(0, part)
    score.write(fp="output.musicxml")

In [142]:
def main():
    export()

if __name__ == "__main__":
    main()
